In [4]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier

BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR))
from notebooks.preprocessing import CreditPreprocessor

DATA_DIR, MODELS_DIR = BASE_DIR / "data", BASE_DIR / "models"


In [5]:
cfg = json.loads((MODELS_DIR / "threshold.json").read_text())

model = XGBClassifier()
model.load_model(MODELS_DIR / "xgb.json")
calibrator = joblib.load(MODELS_DIR / "calibrator.pkl")
preprocessor = CreditPreprocessor.load(MODELS_DIR / "preprocessor.json")

raw_data = pd.read_csv(DATA_DIR / "test.csv")

print("model      :", cfg["model"], f"({cfg['n_trees']} trees)")
print("threshold  :", round(cfg["threshold"], 3), f"(F1: {cfg['threshold_f1']:.3f} | cost: {cfg['threshold_cost']:.3f})")
print("test.csv   :", raw_data.shape)

model      : XGBoost + isotonic calibration (259 trees)
threshold  : 0.162 (F1: 0.231 | cost: 0.162)
test.csv   : (45195, 11)


In [6]:
test_data = preprocessor.transform(raw_data)

print("NaN values left :", int(test_data.isnull().sum().sum()))

expected = cfg["feature_columns"]
print("Columns match the trained model exactly?", list(test_data.columns) == expected)
assert list(test_data.columns) == expected, "Feature columns differ from training; re-run the modeling notebook"

test_data = test_data[expected]

NaN values left : 0
Columns match the trained model exactly? True


In [7]:
score = model.predict_proba(test_data)[:, 1]
# isotonic output is a step function and can hit exactly 0 or 1 at the extremes; clip to keep it realistic
proba = np.clip(calibrator.predict(score), 0.001, 0.99)
high_risk = proba >= cfg["threshold"]

print(f"Min    : {proba.min():.4f}")
print(f"Max    : {proba.max():.4f}")
print(f"Mean   : {proba.mean():.4f}   (training default rate is about 0.066)")
print(f"Median : {np.median(proba):.4f}")
print(f"\nFlagged high risk: {high_risk.sum()} of {len(proba)} ({high_risk.mean():.1%})")

Min    : 0.0010
Max    : 0.9900
Mean   : 0.0672   (training default rate is about 0.066)
Median : 0.0206

Flagged high risk: 4735 of 45195 (10.5%)


In [8]:
check_df = raw_data.copy()
check_df["Probability"] = proba
check_df["HighRisk"] = high_risk
cols = [c for c in ["Id", "MonthlyIncome", "DebtRatio", "Probability", "HighRisk"] if c in check_df.columns]
display(check_df.sort_values("Probability", ascending=False).head(10)[cols])

,Id,MonthlyIncome,DebtRatio,Probability,HighRisk
26359,87645,5550.0,1.208431,0.990000,True
44450,147593,7351.0,1.466540,0.990000,True
32487,107593,6416.0,1.726819,0.990000,True
20610,68537,6500.0,1.444855,0.990000,True
10610,35178,2000.0,2.941529,0.957027,True
25464,84699,3770.0,1.282418,0.933333,True
13779,45739,8783.0,0.875342,0.933333,True
43874,145585,5000.0,0.907618,0.933333,True
2151,7033,6924.0,0.574440,0.933333,True
12796,42448,9000.0,0.858238,0.930168,True


In [9]:
bins = [0, .02, .05, .1, .2, .3, .5, 1.0001]
dist = pd.cut(pd.Series(proba), bins, right=False).value_counts(sort=False)
display(pd.DataFrame({"borrowers": dist, "share": (dist / len(proba)).round(3)}))

,borrowers,share
"[0.0, 0.02)",20907,0.463
"[0.02, 0.05)",11254,0.249
"[0.05, 0.1)",4593,0.102
"[0.1, 0.2)",4463,0.099
"[0.2, 0.3)",1291,0.029
"[0.3, 0.5)",1697,0.038
"[0.5, 1.0)",990,0.022


In [10]:
submission = pd.DataFrame()
submission["Id"] = raw_data["Id"] if "Id" in raw_data.columns else range(len(proba))
submission["Probability"] = proba

submission.to_csv(DATA_DIR / "xgb_submission.csv", index=False)
display(submission.head())
print("saved:", DATA_DIR / "xgb_submission.csv")

,Id,Probability
0,0,0.517162
1,9,0.031235
2,17,0.013737
3,18,0.027742
4,24,0.010025


saved: d:\CreditSense\data\xgb_submission.csv
